# 03. 사람이 말로 밝힌 선호 vs LLM이 내부에서 읽은 선호 (PRISM)

**가설**: 사람이 "AI가 이랬으면 좋겠다"를 자기 말로 쓸 때, LLM 내부 표상은 그 사람이 뜻한 것과 다를 수 있다. 그 차이는 **언어 형태(부정·강도·hedging)** 에서 올 수 있다.

**데이터**: PRISM 설문 1,500명. 한 사람마다 같은 의사가 두 형태로 있다.
- `system_string`: 자유 서술 ("AI가 이랬으면…", 평균 45단어)
- `stated_prefs`: 같은 사람이 8개 속성 중요도를 0–100으로 매긴 숫자 → **정답**

| 단계 | 하는 일 | 정답 |
|---|---|---|
| 1 | 각 문장을 모델에 넣고 레이어별 hidden state 저장 | — |
| 2 | probe: hidden state → 그 사람의 8개 점수 예측 (vs TF-IDF, 문장 임베딩, 셔플 대조군) | `stated_prefs` |
| 3 | 같은 모델에게 직접 점수를 매기게 함 → "아는 것 vs 말하는 것" | `stated_prefs` |
| 4 | 최소쌍: 강도 사다리(부정 포함) + 실제 문장에 한 문장 추가 → 내부 해석이 형태를 따라가나 | 만들 때 정해진 방향 |
| 5 | 발표용 그림 | — |

> 이 노트북은 `src/`의 스크립트를 순서대로 실행하고 결과를 해석하는 용도야. vast.ai의 Jupyter에서 `llm-personalization/notebooks/`에 두고 실행해.

## 0. 설정

In [ ]:
# vast.ai 처음 한 번만
# !pip install -q -r ../requirements.txt

: 

In [ ]:
import os, sys, json, subprocess
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = os.path.abspath("..")            # llm-personalization/
sys.path.insert(0, os.path.join(ROOT, "src"))
from common import ATTRS, download_survey, load_users

MODEL = "Qwen/Qwen2.5-7B-Instruct"      # Llama 쓰려면 "meta-llama/Llama-3.1-8B-Instruct" + HF_TOKEN
RUN   = os.path.join(ROOT, "runs", "qwen7b")
LIMIT = 0                               # 0 = 전체 1,500명. 빠른 점검용이면 100 정도
SURVEY = download_survey(os.path.join(ROOT, "data", "survey.jsonl"))

def run(script, *args, done=None):
    # src/ 스크립트를 실행하고 마지막 출력만 보여줌. done 파일이 이미 있으면 건너뜀(재실행 시 시간 절약)
    if done and os.path.exists(os.path.join(RUN, done)):
        print(f"[skip] {done} 이미 있음 — 다시 돌리려면 그 파일을 지우세요"); return
    cmd = [sys.executable, os.path.join(ROOT, "src", script), *map(str, args)]
    p = subprocess.run(cmd, cwd=os.path.join(ROOT, "src"), capture_output=True, text=True)
    print("\n".join((p.stdout or "").strip().splitlines()[-25:]))
    if p.returncode != 0:
        err = [l for l in (p.stderr or "").splitlines() if "Loading weights" not in l and "%|" not in l]
        print("returncode", p.returncode); print("\n".join(err[-30:]))
        raise RuntimeError(script + " failed")

# GPU 확인 (커널에 torch를 올리지 않아서 모델 실행 메모리를 아낌)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout or "GPU 없음 (CPU)")

## 1. 데이터 살펴보기

먼저 "정답"인 숫자 평점의 분포부터 본다. **천장 효과**가 있으면(다들 90점 이상) 사람 간 차이가 작아서 예측이 어렵다. 그래서 절대 점수와 함께 **사람 안의 상대 순위**(그 사람 평균 대비)도 본다.

In [ ]:
users = load_users(SURVEY)
print("사용자 수:", len(users), "| system_string 단어 수 중앙값:", int(users.text.str.split().str.len().median()))
desc = users[ATTRS].describe().T[["mean", "std"]].round(1)
desc["% >= 90"] = (users[ATTRS] >= 90).mean().round(2)
desc

In [ ]:
for t in users.sample(3, random_state=1).itertuples():
    print("-", t.text[:250], "\n  →", {a: int(getattr(t, a)) for a in ATTRS}, "\n")

**간단한 키워드 점검**: 글에서 정확성을 언급한 사람이 실제로 factuality 점수를 더 높게 줬나? 차이가 조금이라도 있으면, 자유 서술과 숫자가 같은 의사를 담고 있다는 최소한의 근거가 된다.

In [ ]:
import re
for attr, pat in [("factuality", r"accura|fact|truth|correct"), ("creativity", r"creativ|imaginat|humou?r|fun"),
                  ("safety", r"safe|harm|danger"), ("personalisation", r"personal|tailor|adapt")]:
    m = users.text.str.contains(pat, flags=re.I)
    print(f"{attr:16s} 언급 {m.sum():4d}명: {users.loc[m, attr].mean():.1f}  vs  미언급: {users.loc[~m, attr].mean():.1f}")

## 2. hidden state 추출 (GPU)

각 `system_string`을 `"Here is what I want from an AI assistant: ..."` 형태로 넣고, **사용자가 쓴 단어 구간의 평균**과 **마지막 토큰**을 모든 레이어에서 저장한다. 7B 기준 수 분.

In [ ]:
args = ["--model", MODEL, "--out", RUN, "--survey", SURVEY]
if LIMIT: args += ["--limit", LIMIT]
run("extract_acts.py", *args, done="H_mean.npy")

## 3. Probe: 내부 표상이 그 사람의 숫자 평점을 담고 있나

레이어마다 ridge 회귀(사용자 단위 5-fold), 속성별 Spearman ρ. 비교 기준:
- **TF-IDF / 문장 임베딩**: 단어·문장 의미만으로 얼마나 맞히나
- **셔플 대조군**: 정답을 섞었을 때 (≈0이어야 정상)

**읽는 법**: LLM probe가 TF-IDF·임베딩보다 높으면 모델이 표면 단어 이상을 읽고 있다는 뜻. 속성별로 크게 다르면 그 차이가 "사람이 말한 것과 모델이 읽은 것의 간극" 후보다.

In [ ]:
run("probe.py", "--run", RUN, done="probe_summary.json")
summ = json.load(open(os.path.join(RUN, "probe_summary.json")))
rows = {k: v for k, v in summ.items() if isinstance(v, dict)}
pd.DataFrame(rows).T[ATTRS + ["mean", "layer"]]

## 4. 직접 물어보기: 모델이 '말하는' 해석

같은 모델에게 글을 읽고 8개 속성 점수를 JSON으로 매기게 한다. probe(내부)와 이 결과(말)를 같은 정답으로 비교한다.
- 내부 > 말: 모델이 알긴 아는데 말로 꺼내지 못함
- 말 > 내부: 생성 과정에서 추론을 더 함 (선형 probe가 못 잡는 정보)
- 모든 속성에 비슷하게 높은 점수를 주면 절대 점수 대신 상대 순위(`rel`)를 봐야 한다.

In [ ]:
run("verbal_extract.py", "--model", MODEL, "--run", RUN, done="verbal_summary.json")
verbal = json.load(open(os.path.join(RUN, "verbal_summary.json")))
print("JSON 파싱 성공률:", verbal["parse_rate"])
pd.DataFrame({"verbal_abs": verbal["abs"], "verbal_rel": verbal["rel"],
              "probe_abs": summ["llm_abs"], "probe_rel": summ["llm_rel"]}).loc[ATTRS + ["mean"]]

In [ ]:
v = pd.read_parquet(os.path.join(RUN, "verbal.parquet"))
print(v.raw.iloc[0][:400])

## 5. 최소쌍: 내부 해석이 언어 형태를 따라가나 (NLP 핵심)

**A. 강도 사다리**: 속성마다 같은 내용을 6단계로 말한다.
`essential` > `really want` > `nice if` > `don't care` > `need not` > `do not want`
→ probe로 읽은 점수가 이 순서를 따르나? (Spearman, 1이면 완벽)

**B. 문장 추가**: 실제 사람 문장 끝에 *"Above all, X matters most to me."* 또는 *"Honestly, X does not matter to me at all."*을 붙인다.
→ X 점수가 그 방향으로 움직이나? 다른 7개 속성은 덜 움직여야 한다.

부정문("does not matter")에서 방향이 틀리면 → **부정을 제대로 반영하지 못하는 표상**이라는 구체적 증거.

In [ ]:
run("minimal_pairs.py", "--model", MODEL, "--run", RUN, done="minimal_summary.json")
ms = json.load(open(os.path.join(RUN, "minimal_summary.json")))
print("probe layer:", ms["layer"], ms["pool"])
print("사다리 평균 Spearman:", ms["ladder_mean_spearman"])
print("문장 추가 → 올바른 방향 비율  up:", ms["context_correct_up"], "| down:", ms["context_correct_down"])
pd.DataFrame(ms["ladder"]).T

## 6. 발표용 그림

In [ ]:
run("make_figures.py", "--run", RUN)
for f in sorted(os.listdir(os.path.join(RUN, "figures"))):
    print(f); display(Image(os.path.join(RUN, "figures", f), width=820))

## 7. 발표 정리용 체크리스트

- [ ] 3절: 가장 잘/못 읽히는 속성, 최적 레이어, TF-IDF·임베딩 대비 차이
- [ ] 4절: 내부(probe) vs 말(verbal) 중 어느 쪽이 사람 숫자에 가까운가, 속성별로 뒤집히는 곳
- [ ] 5절: 부정(down)이 긍정(up)보다 덜 맞는가 → "부정에 둔감한 표상" 주장 가능 여부
- [ ] 한계: `stated_prefs` 자체가 잡음 있는 자기보고, 최소쌍은 템플릿 문장, probe는 선형